In [1]:
# This runs the mac command to find the exact path to Java 17
import os
import pyspark.sql.functions as F
from pyspark.sql.functions import window, column, desc, col
java17_path = os.popen("/usr/libexec/java_home -v 17").read().strip()

if java17_path:
    os.environ["JAVA_HOME"] = java17_path
    print(f"✅ JAVA_HOME set to: {java17_path}")
else:
    print("❌ Error: Java 17 not found. Please install it or check path.")

✅ JAVA_HOME set to: /Library/Java/JavaVirtualMachines/jdk-17.jdk/Contents/Home


In [2]:
current_path = os.getcwd()
current_path

'/Users/mustakshaikh/Documents/Python/Spark-The-Definitive-Guide/jupyter-code'

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark Toolset") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/01 19:20:26 WARN Utils: Your hostname, Mushtaq.local, resolves to a loopback address: 127.0.0.1; using 192.168.68.66 instead (on interface en0)
26/01/01 19:20:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/01 19:20:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
import glob
search_path = "../data/retail-data/by-day/*.csv"
files = glob.glob(search_path)
print(f"Found {(files)} files.")
if files:
  staticDataFrame = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load(files)

Found ['../data/retail-data/by-day/2011-03-03.csv', '../data/retail-data/by-day/2011-03-17.csv', '../data/retail-data/by-day/2010-12-19.csv', '../data/retail-data/by-day/2011-11-17.csv', '../data/retail-data/by-day/2011-11-03.csv', '../data/retail-data/by-day/2011-06-27.csv', '../data/retail-data/by-day/2011-01-06.csv', '../data/retail-data/by-day/2011-08-22.csv', '../data/retail-data/by-day/2011-01-12.csv', '../data/retail-data/by-day/2011-01-13.csv', '../data/retail-data/by-day/2011-01-07.csv', '../data/retail-data/by-day/2011-08-23.csv', '../data/retail-data/by-day/2011-06-26.csv', '../data/retail-data/by-day/2011-11-02.csv', '../data/retail-data/by-day/2011-11-16.csv', '../data/retail-data/by-day/2011-03-16.csv', '../data/retail-data/by-day/2011-03-02.csv', '../data/retail-data/by-day/2011-04-21.csv', '../data/retail-data/by-day/2011-03-28.csv', '../data/retail-data/by-day/2011-03-14.csv', '../data/retail-data/by-day/2011-11-28.csv', '../data/retail-data/by-day/2011-11-14.csv', '..

In [5]:
staticDataFrame.createOrReplaceTempView("retail_data")
staticSchema = staticDataFrame.schema
staticSchema

StructType([StructField('InvoiceNo', StringType(), True), StructField('StockCode', StringType(), True), StructField('Description', StringType(), True), StructField('Quantity', IntegerType(), True), StructField('InvoiceDate', TimestampType(), True), StructField('UnitPrice', DoubleType(), True), StructField('CustomerID', DoubleType(), True), StructField('Country', StringType(), True)])

In [6]:
staticDataFrame.groupBy('InvoiceDate').agg(F.count('InvoiceNo')).show()

+-------------------+----------------+
|        InvoiceDate|count(InvoiceNo)|
+-------------------+----------------+
|2011-12-08 13:13:00|               2|
|2011-11-29 12:38:00|               1|
|2011-11-29 16:16:00|              29|
|2011-11-16 17:03:00|              18|
|2011-11-22 15:04:00|              12|
|2011-11-23 11:37:00|              11|
|2011-11-22 09:18:00|              10|
|2011-11-23 12:38:00|               6|
|2011-11-15 12:43:00|               2|
|2011-11-20 10:51:00|               9|
|2011-12-06 10:38:00|               2|
|2011-12-06 12:20:00|               3|
|2011-11-28 16:32:00|               2|
|2011-11-10 13:34:00|              14|
|2011-11-10 15:51:00|              18|
|2011-12-05 13:49:00|              38|
|2011-12-05 17:36:00|              51|
|2011-11-08 15:48:00|              23|
|2011-11-22 10:34:00|               2|
|2011-11-23 17:43:00|              27|
+-------------------+----------------+
only showing top 20 rows


In [7]:
staticDataFrame\
.filter('CustomerId==15031') \
.selectExpr(
"CustomerId",
"(UnitPrice * Quantity) as total_cost",
"InvoiceDate")\
.groupBy(
col("CustomerId"), window(col("InvoiceDate"), "5 day"))\
.sum("total_cost")\
.show(10)

+----------+--------------------+------------------+
|CustomerId|              window|   sum(total_cost)|
+----------+--------------------+------------------+
|   15031.0|{2011-12-01 18:00...|            175.53|
|   15031.0|{2011-11-06 18:00...|230.82000000000002|
|   15031.0|{2011-08-18 19:00...|110.96000000000001|
|   15031.0|{2011-03-01 18:00...|150.82999999999998|
+----------+--------------------+------------------+



In [8]:
staticDataFrame.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)



In [9]:
from pyspark.sql.functions import date_format, col
preppedDataFrame = staticDataFrame\
.na.fill(0)\
.withColumn("day_of_week", date_format(col("InvoiceDate"), "EEEE"))\
.coalesce(5)

In [10]:
preppedDataFrame.show()

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|day_of_week|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+
|   580538|    23084|  RABBIT NIGHT LIGHT|      48|2011-12-05 08:38:00|     1.79|   14075.0|United Kingdom|     Monday|
|   580538|    23077| DOUGHNUT LIP GLOSS |      20|2011-12-05 08:38:00|     1.25|   14075.0|United Kingdom|     Monday|
|   580538|    22906|12 MESSAGE CARDS ...|      24|2011-12-05 08:38:00|     1.65|   14075.0|United Kingdom|     Monday|
|   580538|    21914|BLUE HARMONICA IN...|      24|2011-12-05 08:38:00|     1.25|   14075.0|United Kingdom|     Monday|
|   580538|    22467|   GUMBALL COAT RACK|       6|2011-12-05 08:38:00|     2.55|   14075.0|United Kingdom|     Monday|
|   580538|    21544|SKULLS  WATER TRA..

In [11]:
trainDataFrame = preppedDataFrame\
.where("InvoiceDate < '2011-07-01'")
testDataFrame = preppedDataFrame\
.where("InvoiceDate >= '2011-07-01'")

In [12]:
trainDataFrame.show()

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|day_of_week|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+
|   537226|    22811|SET OF 6 T-LIGHTS...|       6|2010-12-06 08:34:00|     2.95|   15987.0|United Kingdom|     Monday|
|   537226|    21713|CITRONELLA CANDLE...|       8|2010-12-06 08:34:00|      2.1|   15987.0|United Kingdom|     Monday|
|   537226|    22927|GREEN GIANT GARDE...|       2|2010-12-06 08:34:00|     5.95|   15987.0|United Kingdom|     Monday|
|   537226|    20802|SMALL GLASS SUNDA...|       6|2010-12-06 08:34:00|     1.65|   15987.0|United Kingdom|     Monday|
|   537226|    22052|VINTAGE CARAVAN G...|      25|2010-12-06 08:34:00|     0.42|   15987.0|United Kingdom|     Monday|
|   537226|    22705|   WRAP GREEN PEARS

In [13]:
from pyspark.ml.feature import StringIndexer
indexer = StringIndexer()\
.setInputCol("day_of_week")\
.setOutputCol("day_of_week_index")

In [14]:
from pyspark.ml.feature import OneHotEncoder
encoder = OneHotEncoder()\
.setInputCol("day_of_week_index")\
.setOutputCol("day_of_week_encoded")

In [15]:
from pyspark.ml.feature import VectorAssembler
vectorAssembler = VectorAssembler()\
.setInputCols(["UnitPrice", "Quantity", "day_of_week_encoded"])\
.setOutputCol("features")

In [16]:
from pyspark.ml import Pipeline
transformationPipeline = Pipeline()\
.setStages([indexer, encoder, vectorAssembler])

In [17]:
fittedPipeline = transformationPipeline.fit(trainDataFrame)

In [25]:
transformedTraining = fittedPipeline.transform(trainDataFrame)

In [26]:
transformedTraining.show()

+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+-----------------+-------------------+--------------------+
|InvoiceNo|StockCode|         Description|Quantity|        InvoiceDate|UnitPrice|CustomerID|       Country|day_of_week|day_of_week_index|day_of_week_encoded|            features|
+---------+---------+--------------------+--------+-------------------+---------+----------+--------------+-----------+-----------------+-------------------+--------------------+
|   537226|    22811|SET OF 6 T-LIGHTS...|       6|2010-12-06 08:34:00|     2.95|   15987.0|United Kingdom|     Monday|              2.0|      (5,[2],[1.0])|(7,[0,1,4],[2.95,...|
|   537226|    21713|CITRONELLA CANDLE...|       8|2010-12-06 08:34:00|      2.1|   15987.0|United Kingdom|     Monday|              2.0|      (5,[2],[1.0])|(7,[0,1,4],[2.1,8...|
|   537226|    22927|GREEN GIANT GARDE...|       2|2010-12-06 08:34:00|     5.95|   15987.0|United Kingdo

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import VectorAssembler

#preparedTestDataFrame = assembler.transform(testDataFrame)
#fittedTestPipeline = transformationPipeline.fit(testDataFrame)
transformedTestTraining = fittedPipeline.transform(testDataFrame)
# --- STEP 1: Initialization (NO CHANGE NEEDED) ---
# Your existing code is fine (just remember no '1L')
kmeans = KMeans()\
  .setK(20)\
  .setSeed(1)

# --- STEP 2: Training (NO CHANGE NEEDED) ---
# Assuming you are using a Pipeline or fitting directly
kmModel = kmeans.fit(transformedTraining)


# --- STEP 3: Evaluation (THIS IS NEW) ---
# Make predictions on your test data
predictions = kmModel.transform(transformedTestTraining)

# Calculate Silhouette Score instead of ComputeCost
evaluator = ClusteringEvaluator()
silhouette = evaluator.evaluate(predictions)

print(f"Silhouette Score: {silhouette}")

Silhouette Score: 0.5255522862104164


26/01/02 15:19:04 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 139184 ms exceeds timeout 120000 ms
26/01/02 15:19:04 WARN SparkContext: Killing executors is not supported by current scheduler.
26/01/02 15:19:13 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$